# Retail Customer Behavior Segmentation
This notebook demonstrates the process of segmenting retail customers based on their transaction patterns using RFM (Recency, Frequency, Monetary) analysis and K-Means clustering with real-world data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Setting styles
plt.style.use('ggplot')
sns.set_palette('viridis')
%matplotlib inline

## 1. Load Data
We use the real Online Retail dataset.

In [ ]:
df = pd.read_csv('../sample/online_retail.csv', encoding='ISO-8859-1')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Preprocessing: remove rows without CustomerID and negative quantities
df = df.dropna(subset=['CustomerID'])
df = df[df['Quantity'] > 0]

print(f'Total records: {len(df)}')
df.head()

## 2. RFM Calculation
- **Recency**: Days since last purchase.
- **Frequency**: Total number of transactions.
- **Monetary**: Total revenue generated.

In [ ]:
analysis_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (analysis_date - x.max()).days,
    'InvoiceNo': 'count',
    'TotalPrice': 'sum'
})

rfm.rename(columns={'InvoiceDate': 'Recency',
                    'InvoiceNo': 'Frequency',
                    'TotalPrice': 'Monetary'}, inplace=True)
rfm.head()

## 3. Data Preprocessing
Standardizing the features for K-Means.

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

rfm_scaled_df = pd.DataFrame(rfm_scaled, index=rfm.index, columns=rfm.columns)
rfm_scaled_df.head()

## 4. Elbow Method
Identifying the optimal number of clusters.

In [ ]:
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    wcss.append(kmeans.inertia_)

plt.plot(range(1, 11), wcss, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()

## 5. K-Means Clustering
Assigning customers to 4 segments.

In [ ]:
kmeans = KMeans(n_clusters=4, init='k-means++', random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Profile the clusters
cluster_analysis = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean'
}).sort_values('Monetary', ascending=False)
cluster_analysis

## 6. Visualization
Visualizing the segments in 3D.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(rfm['Recency'], rfm['Frequency'], rfm['Monetary'], c=rfm['Cluster'], cmap='viridis', s=40)
ax.set_xlabel('Recency')
ax.set_ylabel('Frequency')
ax.set_zlabel('Monetary')
plt.title('3D Customer Segmentation')
plt.show()